# combine master files

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name=version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        #print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed
This notebook was generated at 2023-04-30 19:25:54 (KST = GMT+0900) 
0 Python     3.9.16 64bit [GCC 11.2.0]
1 IPython    8.12.0
2 OS         Linux 5.15.0 71 generic x86_64 with glibc2.31
3 numpy      1.21.5
4 pandas     1.5.3
5 matplotlib 3.7.1
6 scipy      1.7.3
7 astropy    5.1
8 photutils  1.6.0
9 ccdproc    2.4.0
10 version_information 1.0.4


### import modules

In [2]:
from glob import glob
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

import _astro_utilities
import _Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

In [3]:
#%%
BASEDIR = Path("/mnt/Rdata/CCD_obs") 
#BASEDIR = Path("/mnt/OBS_data") 
DOINGDIR = Path(BASEDIR/ "RnE_2022/GSON300_STF-8300M")
#DOINGDIR = Path(BASEDIR/ "CCD_new_files1")

#DOINGDIRs = sorted(_Python_utilities.getFullnameListOfsubDirs(DOINGDIR))
DOINGDIRs = sorted([x for x in DOINGDIR.iterdir() if x.is_dir()])
#print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

len(DOINGDIRs):  7


In [7]:
for DOINGDIR in DOINGDIRs[:1] :
    #DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        print(f"Starting: {str(DOINGDIR.parts[-1])}")
    
        MASTERDIR = DOINGDIR / _astro_utilities.master_dir
        REDUCEDDIR = DOINGDIR / _astro_utilities.reduced_dir
        SOLVEDDIR = DOINGDIR / _astro_utilities.solved_dir2

        if not SOLVEDDIR.exists():
            os.makedirs("{}".format(str(SOLVEDDIR)))
            print("{} is created...".format(str(SOLVEDDIR)))

        summary = yfu.make_summary(DOINGDIR/"*.fit*")
        print("len(summary):", len(summary))
        print("summary:", summary)
        #print(summary["file"][0])

DOINGDIR /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin
len(fits_in_dir) 263
Starting: KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin
All 45 keywords (guessed from /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_BIAS_-_2022-10-09-11-02-32_1sec_-_STF-8300M_-20c_1bin.fit) will be loaded.


/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key EXTEND not found for /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_DARK_-_2022-11-14-13-05-46_180sec_-_STF-8300M_-19c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key EXPTIME not found for /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_DARK_-_2022-11-14-13-05-46_180sec_-_STF-8300M_-19c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key FWHEEL not found for /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_DARK_-_2022-11-14-13-05-46_180sec_-_STF-8300M_-19c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, 

len(summary): 263
summary:                                                   file  filesize  SIMPLE  \
0    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
1    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
2    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
3    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
4    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
..                                                 ...       ...     ...   
258  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
259  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
260  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
261  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
262  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   

     BITPIX  NAXIS  NAXIS1  NAXIS2 EXTEND    BZERO IMAGETYP 

In [6]:

summary = yfu.make_summary(DOINGDIR/"*.fit*")
print("len(summary):", len(summary))
print("summary:", summary)
#print(summary["file"][0])

All 45 keywords (guessed from /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_BIAS_-_2022-10-09-11-02-32_1sec_-_STF-8300M_-20c_1bin.fit) will be loaded.


/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key FOCNAME not found for /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_BIAS_-_2022-10-26-13-29-51_1sec_-_STF-8300M_17c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key FOCPOS not found for /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_BIAS_-_2022-10-26-13-29-51_1sec_-_STF-8300M_17c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key FOCUSPOS not found for /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/-_BIAS_-_2022-10-26-13-29-51_1sec_-_STF-8300M_17c_1bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(ite

len(summary): 263
summary:                                                   file  filesize  SIMPLE  \
0    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
1    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
2    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
3    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
4    /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16980480    True   
..                                                 ...       ...     ...   
258  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
259  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
260  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
261  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
262  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   

     BITPIX  NAXIS  NAXIS1  NAXIS2 EXTEND    BZERO IMAGETYP 

In [9]:
df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
df_light = df_light.reset_index(drop=True)
print("df_light:\n{}".format(df_light))

df_light:
                                                 file  filesize  SIMPLE  \
0   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
1   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
2   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
3   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
4   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
5   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
6   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
7   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
8   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
9   /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
10  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
11  /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/...  16983360    True   
12  /mnt/Rdata/

In [10]:
#%%
n = 0
for _, row  in df_light.iterrows():

    #fullname = fullnames[5]
    n += 1
    print('#'*40,
        "\n{2:.01f}%  ({0}/{1})".format(n, len(df_light), (n/len(df_light))*100))
    print ("Starting...\nfullname: {}".format(row["file"]))

######################################## 
2.2%  (1/45)
Starting...
fullname: /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/KLEOPATRA_Light_b_2022-10-23-10-49-48_120sec_GSON300_STF-8300M_-19C_1bin.fit
######################################## 
4.4%  (2/45)
Starting...
fullname: /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/KLEOPATRA_Light_b_2022-10-23-11-09-13_120sec_GSON300_STF-8300M_-19C_1bin.fit
######################################## 
6.7%  (3/45)
Starting...
fullname: /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/KLEOPATRA_Light_b_2022-10-23-11-15-27_120sec_GSON300_STF-8300M_-20C_1bin.fit
######################################## 
8.9%  (4/45)
Starting...
fullname: /mnt/Rdata/CCD_obs/RnE_2022/GSON300_STF-8300M/KLEOPATRA_Light_-_2022-10-23_-_GSON300_STF-8300M_-_1bin/KLEOPATRA_Light_b_2022-10-23-12-23-01_180sec_GSON300_

In [11]:
ccd = yfu.load_ccd(row["file"])
dir(ccd)

['__abstractmethods__',
 '__array__',
 '__array_prepare__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_arithmetic',
 '_arithmetic_data',
 '_arithmetic_mask',
 '_arithmetic_meta',
 '_arithmetic_uncertainty',
 '_arithmetic_wcs',
 '_data',
 '_flags',
 '_handle_wcs_slicing_error',
 '_insert_in_metadata_fits_safe',
 '_mask',
 '_meta',
 '_prepare_then_do_arithmetic',
 '_slice',
 '_slice_mask',
 '_slice_uncertainty',
 '_slice_wcs',
 '_uncertainty',
 '_unit',
 '_wcs',
 'add',
 'convert_unit_to',
 'copy',
 'data',
 'divide',
 'dtype',
 'flags',
 'header',
 'known_invalid_fits_unit_strings',
 'mask',
 'meta',
 'multiply',
 'ndim',
 'read',
 '

In [ ]:
ccd.header

SIMPLE  =                    T / C# FITS                                        
BITPIX  =                   16 /                                                
NAXIS   =                    2 / Dimensionality                                 
NAXIS1  =                 3352 /                                                
NAXIS2  =                 2532 /                                                
EXTEND  =                    T / Extensions are permitted                       
BZERO   =                32768 /                                                
IMAGETYP= 'LIGHT'              / Type of exposure                               
EXPOSURE=                100.0 / [s] Exposure duration                          
EXPTIME =                100.0 / [s] Exposure duration                          
DATE-LOC= '2022-10-23T21:18:50.726' / Time of observation (local)               
DATE-OBS= '2022-10-23T12:18:50.726' / Time of observation (UTC)                 
XBINNING=                   

In [12]:
ccd.header['PIXSCALE']
print(f"{ccd.header['PIXSCALE']*0.9:.02f}")

0.84


### Solving

In [17]:
import _astro_utilities
ccd = yfu.load_ccd(row["file"])
#solved = _astro_utilities.AstrometrySolver(row["file"], str(SOLVEDDIR), pixscale = ccd.header['PIXSCALE'])
solved = _astro_utilities.makingAstrometrySH(row["file"])
print(solved)
#solved = _astro_utilities.makingAstrometrySH(row["file"], str(SOLVEDDIR), pixscale = ccd.header['PIXSCALE'])

AttributeError: module '_astro_utilities' has no attribute 'makingAstrometrySH'

In [24]:
#%%
n = 0
for _, row  in df_light.iterrows():

    #fullname = fullnames[5]
    n += 1
    print('#'*40,
        "\n{2:.01f}%  ({0}/{1})".format(n, len(df_light), (n/len(df_light))*100))
    print ("Starting...\nfullname: {}".format(row["file"]))

    #_astro_utilities.KevinSolver(row["file"], solved_dir)
    #_astro_utilities.AstrometrySolver(df_light["file"][0], BASEDIR/solved_dir)
    solved = _astro_utilities.AstrometrySolver(row["file"], str(SOLVEDDIR))

######################################## 
5.0%  (1/20)
Starting...
fullname: /mnt/Rdata/CCD_obs/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit
Starting... 
/mnt/Rdata/CCD_obs/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit
str(SOLVEDDIR): /mnt/Rdata/CCD_obs/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/solved
str(SOLVEDDIR/fpath.name) /mnt/Rdata/CCD_obs/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/solved/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit
b'Reading input file 1 of 1: "/mnt/Rdata/CCD_obs/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit"...\nFound an existing WCS header, will try to verify it.\nExtracting sources...\nsimplexy: found 580 sources.\nSolving...\nWarnin